# 🚀 Collaborative Multi-Agent Resume Screening System

### Complete End-to-End Implementation in a Single Notebook

This notebook implements a state-of-the-art **Multi-Agent Resume Screening & Audit Platform** that combines:
1. **Traditional Machine Learning**: TF-IDF Vectorization & Cosine Similarity for keyword/syntax match baselines.
2. **Multi-Agent LLM Architecture** (powered by Google Gemini):
   - 🔍 **Agent 1: Parser & Extractor**: Extracts structured profile data (skills, experience, education, achievements).
   - ⚖️ **Agent 2: Qualification & Match Evaluator**: Evaluates depth of technical fit, project alignment, and gap analysis.
   - 🛡️ **Agent 3: Fact-Checker & Anti-Hallucination Auditor**: Cross-examines claimed skills against employment tenure to prevent resume exaggeration and AI hallucinations.
   - 👑 **Agent 4: Decision & Scoring Synthesizer**: Produces a calibrated weighted score and final hiring recommendation (`Strong Hire`, `Hire`, `Consider`, `Reject`).
   - 🎯 **Agent 5: Adaptive Interview Question Generator**: Formulates targeted technical and behavioral questions tailored to candidate-specific strengths and gaps.
3. **Visual Analytics & Reporting**: Comparative plots, score distribution tables, and structured JSON exports.


## 1. Setup & Dependencies
Uncomment the pip install command below if running in a fresh environment like Google Colab.

In [ ]:
# !pip install google-generativeai pypdf python-docx python-dotenv pydantic scikit-learn matplotlib pandas

import os
import re
import json
import logging
from typing import List, Dict, Any, Optional, Callable
import pandas as pd
import matplotlib.pyplot as plt
from dotenv import load_dotenv
from pydantic import BaseModel, Field
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.metrics.pairwise import cosine_similarity
import google.generativeai as genai
import pypdf
import docx

print("✅ All dependencies successfully imported!")

## 2. API Configuration & Multi-Model Resilience
Loads `GEMINI_API_KEY` from your `.env` file or environment. Includes automatic model cascading to handle rate limits or deprecations gracefully.

In [ ]:
load_dotenv(override=True)
api_key = os.getenv("GEMINI_API_KEY") or os.getenv("GOOGLE_API_KEY")

if not api_key or api_key == "your_gemini_api_key_here":
    api_key = input("Enter your Gemini API Key (or set GEMINI_API_KEY in .env): ").strip()

genai.configure(api_key=api_key)
print("✅ Gemini API configured.")

PRIMARY_MODEL = "gemini-flash-latest"

def generate_structured_json(
    prompt: str,
    schema: Any,
    preferred_model: str = PRIMARY_MODEL,
    temperature: float = 0.1
) -> Dict[str, Any]:
    """
    Calls Gemini with structured JSON schema enforcement and multi-model fallback.
    """
    candidate_models = [
        preferred_model,
        "gemini-flash-latest",
        "gemini-3-flash-preview",
        "gemini-3.5-flash",
        "gemini-2.0-flash",
        "gemini-pro-latest"
    ]
    # Remove duplicates preserving order
    seen = set()
    ordered_models = [m for m in candidate_models if not (m in seen or seen.add(m))]

    last_error = None
    for model_name in ordered_models:
        try:
            model = genai.GenerativeModel(model_name)
            response = model.generate_content(
                prompt,
                generation_config=genai.GenerationConfig(
                    response_mime_type="application/json",
                    response_schema=schema,
                    temperature=temperature
                )
            )
            raw_text = response.text.strip()
            raw_text = re.sub(r'^```json\s*', '', raw_text, flags=re.IGNORECASE)
            raw_text = re.sub(r'^```\s*', '', raw_text)
            raw_text = re.sub(r'\s*```$', '', raw_text)
            return json.loads(raw_text)
        except Exception as e:
            last_error = e
            continue
            
    raise RuntimeError(f"All Gemini models failed. Last error: {last_error}")

## 3. Document Parsers (PDF, DOCX, TXT)
Functions to extract clean raw text from uploaded resume documents.

In [ ]:
def parse_pdf(file_path: str) -> str:
    """Extract text from a PDF document."""
    text = ""
    try:
        reader = pypdf.PdfReader(file_path)
        for page in reader.pages:
            content = page.extract_text()
            if content:
                text += content + "\n"
    except Exception as e:
        text = f"Error reading PDF: {e}"
    return text

def parse_docx(file_path: str) -> str:
    """Extract text from a DOCX file."""
    text = ""
    try:
        doc = docx.Document(file_path)
        for p in doc.paragraphs:
            if p.text:
                text += p.text + "\n"
    except Exception as e:
        text = f"Error reading DOCX: {e}"
    return text

def parse_txt(file_path: str) -> str:
    """Extract text from a plain TXT file."""
    try:
        with open(file_path, 'r', encoding='utf-8', errors='ignore') as f:
            return f.read()
    except Exception as e:
        return f"Error reading TXT: {e}"

def extract_text_from_file(file_path: str) -> str:
    ext = os.path.splitext(file_path)[1].lower()
    if ext == ".pdf":
        return parse_pdf(file_path)
    elif ext in [".docx", ".doc"]:
        return parse_docx(file_path)
    elif ext == ".txt":
        return parse_txt(file_path)
    else:
        return parse_txt(file_path)

## 4. Machine Learning Baseline: TF-IDF & Cosine Similarity
Computes an objective, algorithmic text-overlap score between the Job Description and the Resume.

In [ ]:
def compute_tfidf_similarity(jd_text: str, resume_text: str) -> float:
    """
    Computes Cosine Similarity between TF-IDF representations of the JD and Resume.
    Returns a percentage score from 0.0 to 100.0.
    """
    if not jd_text.strip() or not resume_text.strip():
        return 0.0
    try:
        vectorizer = TfidfVectorizer(stop_words='english')
        tfidf_matrix = vectorizer.fit_transform([jd_text, resume_text])
        similarity = cosine_similarity(tfidf_matrix[0:1], tfidf_matrix[1:2])[0][0]
        return round(float(similarity) * 100, 2)
    except Exception as e:
        print(f"TF-IDF warning: {e}")
        return 0.0

## 5. Pydantic Schemas for Structured Agent Interoperability
Defines strongly-typed contracts for each agent's input and output to eliminate hallucinations and schema drift.

In [ ]:
class ParsedResume(BaseModel):
    candidate_name: str = Field(description="Full name of candidate")
    contact_info: Dict[str, str] = Field(description="Extracted email, phone, location, linkedin, github")
    total_years_experience: float = Field(description="Estimated total professional years of experience")
    technical_skills: List[str] = Field(description="List of programming languages, frameworks, cloud, tools")
    soft_skills: List[str] = Field(description="Communication, leadership, teamwork, etc.")
    work_history: List[Dict[str, str]] = Field(description="Role, Company, Duration, Key Highlights")
    education: List[Dict[str, str]] = Field(description="Degree, Institution, Year")
    certifications: List[str] = Field(description="Professional certifications, courses")

class EvaluationResult(BaseModel):
    skills_match_score: float = Field(description="0-100 match against required technical skills")
    experience_fit_score: float = Field(description="0-100 fit of candidate's past seniority and domain")
    education_fit_score: float = Field(description="0-100 fit of academic background")
    matched_skills: List[str] = Field(description="Required skills explicitly present")
    missing_critical_skills: List[str] = Field(description="Required skills absent from resume")
    evaluation_rationale: str = Field(description="Concise 2-3 sentence technical justification")

class AuditVerification(BaseModel):
    authenticity_confidence: float = Field(description="0-100 confidence score on claims authenticity")
    exaggerated_claims: List[str] = Field(description="Skills listed without supporting projects or tenure")
    employment_timeline_flags: List[str] = Field(description="Timeline inconsistencies, gaps, or overlaps")
    score_adjustment_penalty: float = Field(description="0 to 20 penalty points to deduct from score")
    audit_notes: str = Field(description="Detailed reasoning of the audit verification")

class FinalDecision(BaseModel):
    overall_score: float = Field(description="0-100 calibrated final score")
    verdict: str = Field(description="Strong Hire | Hire | Consider | Reject")
    executive_summary: str = Field(description="High-level hiring manager summary")
    strengths: List[str] = Field(description="Top 3 distinct candidate strengths")
    areas_of_concern: List[str] = Field(description="Top 2-3 risks, gaps, or areas needing probing")

class InterviewQuestions(BaseModel):
    technical_probing_questions: List[str] = Field(description="3 deep questions probing skills or gaps")
    behavioral_questions: List[str] = Field(description="2 behavioral questions based on work history")
    verification_questions: List[str] = Field(description="2 questions verifying timeline or claims")

## 6. Multi-Agent Pipeline Implementation
Here each specialized agent handles its specific domain:

In [ ]:
# ---------------------------------------------------------
# Agent 1: Parser & Extractor Agent
# ---------------------------------------------------------
def agent_resume_parser(resume_text: str, model_name: str = PRIMARY_MODEL) -> ParsedResume:
    prompt = f"""
    You are an Expert Resume Parsing Agent.
    Parse the following resume text into a structured, accurate profile representation.
    Extract total years of experience, technical skills, soft skills, work history, education, and contact details.
    Do not hallucinate or invent skills not mentioned in the text.

    Resume Text:
    """{resume_text[:4000]}"""
    """
    raw = generate_structured_json(prompt, ParsedResume, preferred_model=model_name)
    return ParsedResume(**raw)

# ---------------------------------------------------------
# Agent 2: Qualifications & Match Evaluator
# ---------------------------------------------------------
def agent_evaluator(jd_text: str, parsed: ParsedResume, model_name: str = PRIMARY_MODEL) -> EvaluationResult:
    prompt = f"""
    You are a Senior Technical Recruiter & Qualifications Evaluator.
    Objectively evaluate how well this candidate matches the target Job Description (JD).
    
    Job Description:
    """{jd_text}"""
    
    Candidate Profile:
    {parsed.model_dump_json(indent=2)}
    
    Provide detailed scores (0-100) for skills match, experience fit, education fit, and identify matched & missing skills.
    """
    raw = generate_structured_json(prompt, EvaluationResult, preferred_model=model_name)
    return EvaluationResult(**raw)

# ---------------------------------------------------------
# Agent 3: Fact-Checker & Anti-Hallucination Auditor
# ---------------------------------------------------------
def agent_auditor(raw_resume_text: str, parsed: ParsedResume, eval_res: EvaluationResult, model_name: str = PRIMARY_MODEL) -> AuditVerification:
    prompt = f"""
    You are a Rigorous Resume Auditor & Anti-Hallucination Fact-Checker.
    Your job is to protect hiring managers from inflated resumes and false claims.
    Cross-check the evaluated claims against the raw resume text:
    1. Are skills mentioned as standalone buzzwords with no project tenure or context?
    2. Are there timeline gaps or suspicious overlapping dates?
    3. Assign an authenticity confidence score (0-100) and a penalty deduction (0-20 points) if claims are unverified.
    
    Raw Resume Text:
    """{raw_resume_text[:4000]}"""
    
    Parsed Skills: {parsed.technical_skills}
    Matched Skills: {eval_res.matched_skills}
    """
    raw = generate_structured_json(prompt, AuditVerification, preferred_model=model_name)
    return AuditVerification(**raw)

# ---------------------------------------------------------
# Agent 4: Decision & Scoring Synthesizer
# ---------------------------------------------------------
def agent_decision_synthesizer(
    jd_text: str,
    parsed: ParsedResume,
    eval_res: EvaluationResult,
    audit_res: AuditVerification,
    tfidf_score: float,
    model_name: str = PRIMARY_MODEL
) -> FinalDecision:
    prompt = f"""
    You are the VP of Engineering / Talent Lead making the final hiring recommendation.
    Synthesize all inputs into a balanced decision:
    - TF-IDF Keyword Match Score: {tfidf_score}%
    - Evaluator Skills Score: {eval_res.skills_match_score}%
    - Evaluator Experience Fit: {eval_res.experience_fit_score}%
    - Auditor Confidence: {audit_res.authenticity_confidence}%
    - Auditor Penalty: -{audit_res.score_adjustment_penalty} points
    
    Calculate a fair, calibrated overall score (0-100) taking into account penalties.
    Assign a verdict: 'Strong Hire', 'Hire', 'Consider', or 'Reject'.
    Write an executive summary, top 3 strengths, and key risks.
    """
    raw = generate_structured_json(prompt, FinalDecision, preferred_model=model_name)
    return FinalDecision(**raw)

# ---------------------------------------------------------
# Agent 5: Tailored Interview Questions Generator
# ---------------------------------------------------------
def agent_interview_generator(
    jd_text: str,
    parsed: ParsedResume,
    eval_res: EvaluationResult,
    audit_res: AuditVerification,
    model_name: str = PRIMARY_MODEL
) -> InterviewQuestions:
    prompt = f"""
    You are a Technical Interview Designer.
    Generate 7 highly targeted interview questions tailored specifically to this candidate:
    - 3 Deep Technical questions (probe missing skills or core JD requirements)
    - 2 Behavioral questions (based on their past company tenure and roles)
    - 2 Verification questions (probing any flags or ungrounded claims identified by auditor: {audit_res.exaggerated_claims})
    
    Target Role:
    """{jd_text[:1000]}"""
    Candidate: {parsed.candidate_name}
    """
    raw = generate_structured_json(prompt, InterviewQuestions, preferred_model=model_name)
    return InterviewQuestions(**raw)

## 7. Master Screening Orchestrator
Orchestrates the entire pipeline for a single resume or an entire batch of resumes.

In [ ]:
def screen_resume_pipeline(file_path: str, jd_text: str, model_name: str = PRIMARY_MODEL) -> Dict[str, Any]:
    """
    Executes full multi-agent screening on a single resume file.
    """
    filename = os.path.basename(file_path)
    print(f"\n🔍 [1/5] Extracting text from: {filename}...")
    raw_text = extract_text_from_file(file_path)
    
    print(f"📊 [2/5] Calculating TF-IDF Cosine Similarity...")
    tfidf_score = compute_tfidf_similarity(jd_text, raw_text)
    
    print(f"🤖 [3/5] Agent 1: Parsing resume profile with Gemini...")
    parsed = agent_resume_parser(raw_text, model_name=model_name)
    
    print(f"⚖️  [4/5] Agent 2 & 3: Evaluating qualification & running auditor verification...")
    evaluation = agent_evaluator(jd_text, parsed, model_name=model_name)
    audit = agent_auditor(raw_text, parsed, evaluation, model_name=model_name)
    
    print(f"👑 [5/5] Agent 4 & 5: Synthesizing final decision & generating interview questions...")
    decision = agent_decision_synthesizer(jd_text, parsed, evaluation, audit, tfidf_score, model_name=model_name)
    questions = agent_interview_generator(jd_text, parsed, evaluation, audit, model_name=model_name)
    
    return {
        "filename": filename,
        "candidate_name": parsed.candidate_name,
        "tfidf_score": tfidf_score,
        "overall_score": decision.overall_score,
        "verdict": decision.verdict,
        "years_experience": parsed.total_years_experience,
        "skills_match_score": evaluation.skills_match_score,
        "experience_fit_score": evaluation.experience_fit_score,
        "audit_confidence": audit.authenticity_confidence,
        "audit_penalty": audit.score_adjustment_penalty,
        "matched_skills": evaluation.matched_skills,
        "missing_skills": evaluation.missing_critical_skills,
        "executive_summary": decision.executive_summary,
        "strengths": decision.strengths,
        "areas_of_concern": decision.areas_of_concern,
        "interview_questions": questions.model_dump()
    }

## 8. Run Screening on Real Sample Resumes
Define a Job Description and screen sample candidate resumes from the `sample_resumes/` folder.

In [ ]:
# Target Role Requirements (Job Description)
JOB_DESCRIPTION = """
Role: Senior Full-Stack Software Engineer (Python & React)
Experience: 5+ years

Key Responsibilities & Requirements:
- Strong proficiency in Python, FastAPI, and asynchronous design patterns.
- Hands-on frontend development experience with React and TypeScript.
- Solid database expertise with PostgreSQL, Redis, and schema design.
- Production cloud architecture experience with AWS (ECS, S3, RDS) and Docker containerization.
- Deep commitment to CI/CD automation, unit testing, and maintainable clean architecture.

Nice to Have:
- Practical exposure to LLM integrations, LangChain, or AI agent pipelines.
"""

# Discover candidate resumes in sample_resumes/
sample_dir = "sample_resumes"
resume_files = []
if os.path.exists(sample_dir):
    resume_files = [os.path.join(sample_dir, f) for f in os.listdir(sample_dir) if f.endswith((".txt", ".pdf", ".docx"))]

print(f"Found {len(resume_files)} candidate resumes in '{sample_dir}'.")
# Process up to the first 3 resumes for demonstration
batch_to_screen = resume_files[:3] if resume_files else []

results = []
for r_path in batch_to_screen:
    try:
        res = screen_resume_pipeline(r_path, JOB_DESCRIPTION)
        results.append(res)
    except Exception as e:
        print(f"❌ Error screening {r_path}: {e}")

print(f"\n🎉 Successfully completed screening for {len(results)} candidate(s)!")

## 9. Tabular Comparison & Analytics (Pandas DataFrame)
Formats screening results into an easy-to-read recruiter summary table.

In [ ]:
if results:
    df = pd.DataFrame([
        {
            "Candidate": r["candidate_name"],
            "Final Score": r["overall_score"],
            "Verdict": r["verdict"],
            "TF-IDF Match (%)": r["tfidf_score"],
            "Skills Match (%)": r["skills_match_score"],
            "Experience Fit (%)": r["experience_fit_score"],
            "Auditor Confidence (%)": r["audit_confidence"],
            "Penalty": r["audit_penalty"]
        }
        for r in results
    ])
    # Sort by overall score descending
    df = df.sort_values(by="Final Score", ascending=False).reset_index(drop=True)
    display(df)
else:
    print("No screening results available yet. Run Cell 8 above.")

## 10. Visual Analytics & Decision Plots
Generates comparative bar charts comparing traditional TF-IDF keyword scores vs Multi-Agent LLM calibrated scores.

In [ ]:
if results:
    candidates = [r["candidate_name"] for r in results]
    tfidf_scores = [r["tfidf_score"] for r in results]
    overall_scores = [r["overall_score"] for r in results]
    skills_scores = [r["skills_match_score"] for r in results]

    x = range(len(candidates))
    width = 0.25

    plt.figure(figsize=(10, 6))
    plt.bar([i - width for i in x], tfidf_scores, width=width, label='TF-IDF Score', color='#38bdf8')
    plt.bar(x, skills_scores, width=width, label='Skills Match', color='#818cf8')
    plt.bar([i + width for i in x], overall_scores, width=width, label='Final Overall Score', color='#34d399')

    plt.xlabel('Candidates', fontweight='bold')
    plt.ylabel('Score (0 - 100)', fontweight='bold')
    plt.title('Multi-Agent Candidate Evaluation Comparison', fontsize=14, fontweight='bold')
    plt.xticks(x, candidates)
    plt.ylim(0, 105)
    plt.legend()
    plt.grid(axis='y', linestyle='--', alpha=0.3)
    plt.tight_layout()
    plt.show()
    
    # Detailed view of top candidate
    top = results[0]
    print("=" * 60)
    print(f"🌟 Top Candidate Summary: {top['candidate_name']} ({top['verdict']})")
    print(f"Overall Calibrated Score: {top['overall_score']}/100")
    print(f"Executive Summary:\n{top['executive_summary']}")
    print("\nKey Strengths:")
    for s in top['strengths']:
        print(f"  • {s}")
    print("\nAreas of Concern / Probing:")
    for c in top['areas_of_concern']:
        print(f"  • {c}")
    print("=" * 60)
else:
    print("No results to plot.")